# Fetal Health Analysis

Fetal and maternal health during birth are one of the main concerns of United Nations. Global maternal and fetal health outcomes have drastically improved in the last decades, however stark regional and socioeconomic disparities persist. According to WHO and UNICEF, most maternal deaths during childbirth are preventable with timely access to skilled obstetric care. Additionally data from UN IGME highlights that the period during birth and the first month after carries the highest risk for the fetus and newborn. This project strives to find which health markers have the biggest significance for the outcome of labour.


## Main hypothesis

Fetal heart rate markers (low variability, prolonged and late decelerations) differ significantly between normal and abnormal groups in both datasets, and the strength of these differences is greater than for maternal factors such as age and parity.


In [2]:
%matplotlib inline

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import wfdb
from pathlib import Path

The first dataset is CTU-UHB Intrapartum Cardiotocography Database - a collection of 552 CTGs from the Czech Technical University (CTU) in Prague and the University Hospital in Brno (UHB). *Each recording is up to 90 minutes long, and includes a fetal heart rate time series, a uterine contraction signal, and maternal, delivery, and fetal clinical details.*

In [12]:
data_dir = Path("data/ctu-uhb-ctgdb-1.0.0")
names = sorted(p.stem for p in data_dir.glob("*.hea"))
rows = []
for name in names:
    h = wfdb.rdheader(str(data_dir / name))
    info = {"record": name}
    for line in h.comments:
        line = line.lstrip("#").strip()
        if not line or line.startswith(("-", "!")):
            continue
        key, value = line.rsplit(None, 1)
        try:
            info[key] = float(value)
        except ValueError:
            info[key] = value
    rows.append(info)
df_clinical = pd.DataFrame(rows)

In [13]:
df_clinical

,record,pH,BDecf,pCO2,BE,Apgar1,Apgar5,NICU days,Seizures,HIE,...,Induced,I.stage,NoProgress,CK/KP,II.stage,Deliv. type,dbID,Rec. type,Pos. II.st.,Sig2Birth
0,1001,7.14,8.14,7.7,-10.5,6.0,8.0,0.0,0.0,0.0,...,0.0,232.0,0.0,0.0,20.0,1.0,974204.0,1.0,14400.0,0.0
1,1002,7.00,7.92,12.0,-12.0,8.0,8.0,0.0,0.0,0.0,...,0.0,220.0,0.0,0.0,20.0,1.0,975666.0,1.0,14400.0,0.0
2,1003,7.20,3.03,8.3,-5.6,7.0,9.0,0.0,0.0,0.0,...,1.0,215.0,0.0,0.0,15.0,1.0,982315.0,1.0,14400.0,0.0
3,1004,7.30,5.19,5.5,-6.4,8.0,9.0,0.0,0.0,0.0,...,1.0,55.0,0.0,0.0,10.0,1.0,983277.0,12.0,14400.0,0.0
4,1005,7.30,4.52,5.7,-5.8,9.0,10.0,0.0,0.0,0.0,...,1.0,114.0,0.0,0.0,15.0,1.0,983398.0,1.0,14400.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
547,2042,7.33,-0.50,6.6,-0.8,9.0,10.0,0.0,0.0,0.0,...,0.0,255.0,0.0,1.0,15.0,2.0,1072108.0,1.0,15482.0,0.0
548,2043,7.08,10.92,7.9,-13.3,8.0,9.0,0.0,0.0,0.0,...,0.0,260.0,0.0,1.0,25.0,2.0,1078408.0,1.0,14400.0,0.0
549,2044,7.02,9.13,10.6,-12.3,8.0,8.0,0.0,0.0,0.0,...,1.0,335.0,1.0,0.0,-1.0,2.0,1079432.0,1.0,-1.0,0.0
550,2045,7.03,8.91,10.4,-12.2,7.0,9.0,0.0,0.0,0.0,...,0.0,189.0,0.0,0.0,-1.0,2.0,1083243.0,1.0,-1.0,0.0


In [14]:
df_clinical.to_csv("clinical_ctu_chb.csv")

In [15]:
df_clinical.columns

Index(['record', 'pH', 'BDecf', 'pCO2', 'BE', 'Apgar1', 'Apgar5', 'NICU days',
       'Seizures', 'HIE', 'Intubation', 'Main diag.', 'Other diag.',
       'Gest. weeks', 'Weight(g)', 'Sex', 'Age', 'Gravidity', 'Parity',
       'Diabetes', 'Hypertension', 'Preeclampsia', 'Liq. praecox', 'Pyrexia',
       'Meconium', 'Presentation', 'Induced', 'I.stage', 'NoProgress', 'CK/KP',
       'II.stage', 'Deliv. type', 'dbID', 'Rec. type', 'Pos. II.st.',
       'Sig2Birth'],
      dtype='str')

Since the values in the dataframe have been extracting form a raw text file, containing no information about their format, the first step would be to transform them in the correct type.

In [17]:
cat_cols = ["Presentation", "Deliv. type", "Sex", "Diabetes",
            "Hypertension", "Preeclampsia", "Induced", "Meconium", "Rec. type"]
for c in cat_cols:
    df_clinical[c] = df_clinical[c].astype("category")

Numerical data can be turned into meaningful content in respect to its corresponding code.

In [18]:
df_clinical["Sex"] = df_clinical["Sex"].map({"female": 1, "male": 2})
df_clinical["Presentation"] = df_clinical["Presentation"].map({"normal": 1, "breech": 2, "other": 3})
df_clinical["Deliv. type"] = df_clinical["Deliv. type"].map({"vaginal": 1, "cs": 2})
df_clinical["Rec. type"] = df_clinical["Rec. type"].map({"USG": 1, "FECG": 2, "combined": 12})